## Implementing the Models

In [1]:
import os
import pandas as pd
import numpy as np

path = r"C:\Users\nisha\Desktop\2026 PS-II\Projects\Loan\scientificProject\data"
import_file = os.path.join(path, 'final_data.csv')

Data=pd.read_csv(import_file)
Data=Data.dropna(axis=0,how='any')
print(Data)

      Unnamed: 0  credit.policy  int.rate  installment  log.annual.inc    dti  \
0              0              1    0.1189       829.10       11.350407  19.48   
1              1              1    0.1071       228.22       11.082143  14.29   
2              2              1    0.1357       366.86       10.373491  11.63   
3              3              1    0.1008       162.34       11.350407   8.10   
4              4              1    0.1426       102.92       11.299732  14.97   
...          ...            ...       ...          ...             ...    ...   
9573        9573              0    0.1461       344.76       12.180755  10.39   
9574        9574              0    0.1253       257.70       11.141862   0.21   
9575        9575              0    0.1071        97.81       10.596635  13.09   
9576        9576              0    0.1600       351.58       10.819778  19.18   
9577        9577              0    0.1392       853.43       11.264464  16.28   

      fico  days.with.cr.li

In [2]:
# Train–Test Split

from sklearn.model_selection import train_test_split

# Separate features and target
X = Data.drop('not.fully.paid', axis=1)
y = Data['not.fully.paid']

# Perform the split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Show shapes to verify split correctness
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((6704, 15), (2874, 15), (6704,), (2874,))

In [3]:
from sklearn.preprocessing import StandardScaler

# 1. Identify numerical columns
numeric_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_cols

['Unnamed: 0',
 'credit.policy',
 'int.rate',
 'installment',
 'log.annual.inc',
 'dti',
 'fico',
 'days.with.cr.line',
 'revol.util']

### Implementing Logistic Regression

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# 1. Train Logistic Regression model
log_model = LogisticRegression(max_iter=500)
log_model.fit(X_train, y_train)

# 2. Predict on test set
y_pred_log = log_model.predict(X_test)

# 3. Evaluation metrics
log_accuracy = accuracy_score(y_test, y_pred_log)
log_precision = precision_score(y_test, y_pred_log)
log_recall = recall_score(y_test, y_pred_log)
log_f1 = f1_score(y_test, y_pred_log)

print("----- Logistic Regression Results -----")
print(f"Accuracy       : {log_accuracy:.4f}")
print(f"Precision      : {log_precision:.4f}")
print(f"Recall         : {log_recall:.4f}")
print(f"F1-score       : {log_f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

----- Logistic Regression Results -----
Accuracy       : 0.8399
Precision      : 0.5000
Recall         : 0.0109
F1-score       : 0.0213

Classification Report:
              precision    recall  f1-score   support

           0       0.84      1.00      0.91      2414
           1       0.50      0.01      0.02       460

    accuracy                           0.84      2874
   macro avg       0.67      0.50      0.47      2874
weighted avg       0.79      0.84      0.77      2874


Confusion Matrix:
[[2409    5]
 [ 455    5]]


C:\Users\nisha\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### Adding Random Forest

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# 1. Train Random Forest Model
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    random_state=42
)
rf_model.fit(X_train, y_train)

# 2. Predict on test set
y_pred_rf = rf_model.predict(X_test)

# 3. Evaluation Metrics
rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)

print("----- Random Forest Results -----")
print(f"Accuracy       : {rf_accuracy:.4f}")
print(f"Precision      : {rf_precision:.4f}")
print(f"Recall         : {rf_recall:.4f}")
print(f"F1-score       : {rf_f1:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

----- Random Forest Results -----
Accuracy       : 0.8399
Precision      : 0.5000
Recall         : 0.0348
F1-score       : 0.0650

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.99      0.91      2414
           1       0.50      0.03      0.07       460

    accuracy                           0.84      2874
   macro avg       0.67      0.51      0.49      2874
weighted avg       0.79      0.84      0.78      2874


Confusion Matrix:
[[2398   16]
 [ 444   16]]


### Adding XGBoost

In [6]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# 1) Train Model
xgb_model = XGBClassifier(random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# 2) Predictions
y_pred_xgb = xgb_model.predict(X_test)

# 3) Evaluation
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
xgb_precision = precision_score(y_test, y_pred_xgb, zero_division=0)
xgb_recall = recall_score(y_test, y_pred_xgb)
xgb_f1 = f1_score(y_test, y_pred_xgb)

print("----- XGBoost Classifier Results -----")
print("Accuracy      :", round(xgb_accuracy, 4))
print("Precision     :", round(xgb_precision, 4))
print("Recall        :", round(xgb_recall, 4))
print("F1-score      :", round(xgb_f1, 4))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_xgb))

----- XGBoost Classifier Results -----
Accuracy      : 0.8173
Precision     : 0.2441
Recall        : 0.0674
F1-score      : 0.1056

Classification Report:

              precision    recall  f1-score   support

           0       0.84      0.96      0.90      2414
           1       0.24      0.07      0.11       460

    accuracy                           0.82      2874
   macro avg       0.54      0.51      0.50      2874
weighted avg       0.75      0.82      0.77      2874


Confusion Matrix:

[[2318   96]
 [ 429   31]]
